In [ ]:
!pip -q install transformers accelerate datasets openai wordfreq pandas

In [ ]:
import os

# Backend: "hf" or "cloud"
BACKEND = "hf"

# Hugging Face model
HF_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# Cloud model settings
CLOUD_MODEL_NAME = "api-gpt-oss-120b"
CLOUD_BASE_URL = "https://tritonai-api.ucsd.edu"
# Set your key if using cloud backend
# os.environ["OPENAI_API_KEY"] = "YOUR_KEY"

# Benchmark settings
GAME = "both"          # "spelling_bee", "wordle", "both"
NUM_PUZZLES = 50
MAX_WORDLE_STEPS = 6
MAX_SB_STEPS = 10

USE_THINKING_FORMAT = False
TEMPERATURE = 0.0
MAX_NEW_TOKENS = 16

OUTPUT_DIR = "game_outputs"
RUN_NAME = "open_generation"

In [ ]:
import os
import re
import sys
import json
import math
import random
from pathlib import Path

import pandas as pd
from wordfreq import zipf_frequency

In [ ]:
REPO_URL = "https://github.com/jackiepiepkorn/cse291-nytgames.git"
REPO_DIR = "/content/cse291-nytgames" if os.path.exists("/content") else "./cse291-nytgames"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from nytgames import (
    SpellingBeeConfig,
    SpellingBeeEnv,
    SpellingBeeDataset,
    WordleConfig,
    WordleEnv,
    load_dictionary,
)
from nytgames.env.wordle import _format_feedback

Already up to date.


In [ ]:
def ensure_dir(path: str):
    Path(path).mkdir(parents=True, exist_ok=True)

def extract_first_word(text: str) -> str:
    text = text.strip().lower()
    text = text.replace("*", "").replace('"', "").replace("'", "")
    words = re.findall(r"[a-z]+", text)
    return words[0] if words else ""

def summarize_results(df: pd.DataFrame) -> dict:
    if df.empty:
        return {"rows": 0}

    if "benchmark" not in df.columns:
        return {
            "rows": int(len(df)),
            "columns": list(df.columns),
            "warning": "No benchmark column found."
        }

    summary = {
        "rows": int(len(df)),
        "benchmarks": sorted(df["benchmark"].dropna().unique().tolist()),
        "per_benchmark": {}
    }

    for benchmark in summary["benchmarks"]:
        sub = df[df["benchmark"] == benchmark]
        entry = {"rows": int(len(sub))}

        if benchmark == "spelling_bee":
            if "score" in sub.columns:
                entry["avg_score"] = float(sub["score"].mean())
                entry["max_score"] = float(sub["score"].max())
            if "found" in sub.columns:
                entry["avg_found"] = float(sub["found"].mean())
            if "guess_ratio" in sub.columns:
                entry["avg_guess_ratio"] = float(sub["guess_ratio"].mean())

        elif benchmark == "wordle":
            if "won" in sub.columns:
                entry["win_rate"] = float(sub["won"].mean())
            if "guesses_used" in sub.columns:
                entry["avg_guesses_used"] = float(sub["guesses_used"].mean())

        summary["per_benchmark"][benchmark] = entry

    return summary

def write_outputs(df: pd.DataFrame, out_prefix: str):
    ensure_dir(str(Path(out_prefix).parent))
    csv_path = f"{out_prefix}.csv"
    json_path = f"{out_prefix}.summary.json"
    df.to_csv(csv_path, index=False)
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(summarize_results(df), f, indent=2)
    print("Saved:", csv_path)
    print("Saved:", json_path)

In [ ]:
import re

def parse_spelling_bee_prompt_item(item):
    prompt_msgs = item["prompt"]
    user_text = ""
    for msg in prompt_msgs:
        if msg.get("role") == "user":
            user_text = msg.get("content", "")
            break

    letters_match = re.search(r"Allowed letters:\s*([A-Z,\s]+)", user_text)
    center_match = re.search(r"Center letter .*:\s*([A-Z])", user_text)

    if not letters_match or not center_match:
        raise ValueError(f"Could not parse Spelling Bee prompt:\n{user_text}")

    letters = [x.strip().lower() for x in letters_match.group(1).split(",") if x.strip()]
    center = center_match.group(1).lower()

    return letters, center, user_text

def extract_wordle_guess(text: str) -> str:
    text = text.strip().lower()
    text = text.replace("*", "").replace('"', "").replace("'", "")
    words = re.findall(r"[a-z]+", text)
    for w in words:
        if len(w) == 5:
            return w
    return ""

In [ ]:
class HFBackend:
    def __init__(self, model_name: str):
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM

        self.torch = torch
        self.model_name = model_name

        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.tokenizer.padding_side = "left"
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=dtype,
            device_map="auto",
        )
        self.model.config.pad_token_id = self.tokenizer.pad_token_id

    def generate(self, prompt: str, max_new_tokens: int = 16, temperature: float = 0.0) -> str:
        messages = [
            {
                "role": "system",
                "content": "Reply with exactly one lowercase word and nothing else."
            },
            {
                "role": "user",
                "content": prompt
            },
        ]

        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            padding=True,
            truncation=True,
        )
        inputs = {k: v.to(self.model.device) for k, v in inputs.items()}

        gen_kwargs = {
            "max_new_tokens": 4,
            "do_sample": False,
            "pad_token_id": self.tokenizer.pad_token_id,
            "eos_token_id": self.tokenizer.eos_token_id,
        }

        with self.torch.no_grad():
            outputs = self.model.generate(**inputs, **gen_kwargs)

        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        return self.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


class CloudBackend:
    def __init__(self, model_name: str, base_url: str, api_key: str | None = None):
        from openai import OpenAI

        api_key = api_key or os.environ.get("OPENAI_API_KEY")
        if not api_key:
            raise ValueError("OPENAI_API_KEY not found.")

        self.model_name = model_name
        self.client = OpenAI(api_key=api_key, base_url=base_url)

    def generate(self, prompt: str, max_new_tokens: int = 16, temperature: float = 0.0) -> str:
        resp = self.client.chat.completions.create(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_new_tokens,
            temperature=temperature,
        )
        return resp.choices[0].message.content.strip()

In [ ]:
def make_backend():
    if BACKEND == "hf":
        return HFBackend(HF_MODEL_NAME)
    elif BACKEND == "cloud":
        return CloudBackend(
            model_name=CLOUD_MODEL_NAME,
            base_url=CLOUD_BASE_URL,
            api_key=os.environ.get("OPENAI_API_KEY"),
        )
    else:
        raise ValueError(f"Unknown BACKEND: {BACKEND}")

In [ ]:
def spelling_bee_prompt(center_letter: str, letters: list[str], guessed_words: list[str]) -> str:
    letters_str = ", ".join(sorted(letters))
    guessed_str = ", ".join(guessed_words) if guessed_words else "none"

    return f"""Spelling Bee.

Allowed letters: {letters_str}
Center letter: {center_letter}
Already guessed: {guessed_str}

Return exactly ONE new lowercase word.
Rules:
- at least 4 letters
- must include the center letter
- use only allowed letters
- must not repeat an earlier guess

Output only the word."""


def wordle_prompt(history: list[tuple[str, str]]) -> str:
    hist = "\n".join([f"{g} -> {fb}" for g, fb in history]) if history else "none"

    return f"""Wordle.

History:
{hist}

Return exactly ONE new 5-letter lowercase English word.
Do not repeat any earlier guess.
Output only the word."""

In [ ]:
def evaluate_spelling_bee(backend, num_puzzles=10, repo_dir=REPO_DIR):
    dataset = SpellingBeeDataset()
    results = []

    for i in range(min(num_puzzles, len(dataset))):
        item = dataset[i]

        letters, center, source_prompt = parse_spelling_bee_prompt_item(item)
        all_letters = sorted(set(letters))

        guessed_words = []
        guessed_set = set()
        valid_words = []
        invalid_words = []
        raw_outputs = []

        score = 0

        for step in range(MAX_SB_STEPS):
            prompt = spelling_bee_prompt(center, all_letters, guessed_words)
            raw = backend.generate(
                prompt,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=TEMPERATURE,
            )
            guess = extract_first_word(raw)
            raw_outputs.append(raw)

            if step < 3:
                print(f"[SB puzzle {i} step {step}] raw={raw!r} guess={guess!r}")

            if not guess:
                invalid_words.append("")
                continue

            if guess in guessed_set:
                invalid_words.append(guess)
                continue

            is_valid = True
            if len(guess) < 4:
                is_valid = False
            if center not in guess:
                is_valid = False
            if any(ch not in all_letters for ch in guess):
                is_valid = False

            guessed_set.add(guess)
            guessed_words.append(guess)

            if is_valid:
                valid_words.append(guess)

                word_score = 1 if len(guess) == 4 else len(guess)
                if all(letter in guess for letter in all_letters):
                    word_score += 7
                score += word_score
            else:
                invalid_words.append(guess)

        found = len(valid_words)
        guess_ratio = found / MAX_SB_STEPS if MAX_SB_STEPS > 0 else None

        results.append({
            "benchmark": "spelling_bee",
            "puzzle_idx": i,
            "puzzle_id": item.get("puzzle_id"),
            "score": score,
            "found": found,
            "guess_ratio": guess_ratio,
            "center_letter": center,
            "letters": "".join(all_letters),
            "valid_guesses": valid_words,
            "invalid_guesses": invalid_words,
            "all_guesses": guessed_words,
            "raw_outputs": raw_outputs,
            "source_prompt": source_prompt,
        })

    return pd.DataFrame(results)

In [ ]:
def evaluate_wordle(backend, num_puzzles=10, repo_dir=REPO_DIR):
    dictionary = load_dictionary()
    five_letter_words = sorted(
        [w for w in dictionary if isinstance(w, str) and len(w) == 5 and w.isalpha()]
    )
    five_letter_set = set(five_letter_words)

    rng = random.Random(0)
    sampled_targets = rng.sample(five_letter_words, min(num_puzzles, len(five_letter_words)))

    results = []

    for i, target in enumerate(sampled_targets):
        env = WordleEnv(
            WordleConfig(
                target_word=target,
                word_set=five_letter_set,
            )
        )
        obs, _ = env.reset()

        history = []
        guesses = []
        guessed_set = set()
        won = False
        raw_outputs = []

        for step in range(MAX_WORDLE_STEPS):
            prompt = wordle_prompt(history)
            raw = backend.generate(
                prompt,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=TEMPERATURE,
            )
            guess = extract_wordle_guess(raw)
            raw_outputs.append(raw)

            if step < 3:
                print(f"[WD puzzle {i} step {step}] raw={raw!r} guess={guess!r}")

            if len(guess) != 5:
                continue

            if guess in guessed_set:
                continue

            guessed_set.add(guess)
            guesses.append(guess)

            if guess not in five_letter_set:
                feedback = "invalid"
                history.append((guess, feedback))
                continue

            try:
                obs, reward, terminated, truncated, info = env.step(guess)
            except Exception:
                feedback = _format_feedback(guess, target)
                history.append((guess, feedback))
                continue

            feedback = obs.get("feedback", None) if isinstance(obs, dict) else None
            if feedback is None:
                feedback = _format_feedback(guess, target)

            history.append((guess, feedback))

            if guess == target:
                won = True
                break

            if terminated or truncated:
                break

        results.append({
            "benchmark": "wordle",
            "puzzle_idx": i,
            "target": target,
            "won": won,
            "guesses_used": len(guesses),
            "guesses": guesses,
            "raw_outputs": raw_outputs,
        })

    return pd.DataFrame(results)

In [ ]:
def run_benchmark():
    backend = make_backend()
    frames = []

    if GAME in ("spelling_bee", "both"):
        print("\n=== Running Spelling Bee ===")
        sb_df = evaluate_spelling_bee(backend, num_puzzles=NUM_PUZZLES, repo_dir=REPO_DIR)
        frames.append(sb_df)

    if GAME in ("wordle", "both"):
        print("\n=== Running Wordle ===")
        wd_df = evaluate_wordle(backend, num_puzzles=NUM_PUZZLES, repo_dir=REPO_DIR)
        frames.append(wd_df)

    results_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    out_prefix = str(Path(OUTPUT_DIR) / RUN_NAME)
    write_outputs(results_df, out_prefix)

    summary = summarize_results(results_df)
    return results_df, summary

In [ ]:
results_df, summary = run_benchmark()
summary

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


=== Running Spelling Bee ===
[SB puzzle 0 step 0] raw='warbler' guess='warbler'
[SB puzzle 0 step 1] raw='warblerly' guess='warblerly'
[SB puzzle 0 step 2] raw='warblerly' guess='warblerly'
[SB puzzle 1 step 0] raw='coinfirmed' guess='coinfirmed'
[SB puzzle 1 step 1] raw='coinfirmant' guess='coinfirmant'
[SB puzzle 1 step 2] raw='incomparably' guess='incomparably'
[SB puzzle 2 step 0] raw='celebrity' guess='celebrity'
[SB puzzle 2 step 1] raw='celebrity' guess='celebrity'
[SB puzzle 2 step 2] raw='celebrity' guess='celebrity'
[SB puzzle 3 step 0] raw='triggering' guess='triggering'
[SB puzzle 3 step 1] raw='triggering中心' guess='triggering'
[SB puzzle 3 step 2] raw='triggering中心' guess='triggering'
[SB puzzle 4 step 0] raw='lighter' guess='lighter'
[SB puzzle 4 step 1] raw='lighter' guess='lighter'
[SB puzzle 4 step 2] raw='lighter' guess='lighter'
[SB puzzle 5 step 0] raw='flourish' guess='flourish'
[SB puzzle 5 step 1] raw='flourish' guess='flourish'
[SB puzzle 5 step 2] raw='flouris

{'rows': 100,
 'benchmarks': ['spelling_bee', 'wordle'],
 'per_benchmark': {'spelling_bee': {'rows': 50,
   'avg_score': 0.16,
   'max_score': 5.0,
   'avg_found': 0.08,
   'avg_guess_ratio': 0.008},
  'wordle': {'rows': 50, 'win_rate': 0.0, 'avg_guesses_used': 1.0}}}

In [ ]:
results_df.head()

,benchmark,puzzle_idx,puzzle_id,score,found,guess_ratio,center_letter,letters,valid_guesses,invalid_guesses,all_guesses,raw_outputs,source_prompt,target,won,guesses_used,guesses
0,spelling_bee,0,1.0,0.0,0.0,0.0,w,ahortwy\nc,[],"[warbler, warblerly, warblerly, warblerly, war...","[warbler, warblerly]","[warbler, warblerly, warblerly, warblerly, war...","New Spelling Bee puzzle!\nAllowed letters: A, ...",NaN,NaN,NaN,NaN
1,spelling_bee,1,2.0,0.0,0.0,0.0,i,cfimnor\nc,[],"[coinfirmed, coinfirmant, incomparably, imposs...","[coinfirmed, coinfirmant, incomparably, imposs...","[coinfirmed, coinfirmant, incomparably, imposs...","New Spelling Bee puzzle!\nAllowed letters: C, ...",NaN,NaN,NaN,NaN
2,spelling_bee,2,3.0,0.0,0.0,0.0,c,cefhily\nc,[],"[celebrity, celebrity, celebrity, celebrity, c...",[celebrity],"[celebrity, celebrity, celebrity, celebrity, c...","New Spelling Bee puzzle!\nAllowed letters: C, ...",NaN,NaN,NaN,NaN
3,spelling_bee,3,4.0,0.0,0.0,0.0,t,abdgirt\nc,[],"[triggering, triggering, triggering, triggerin...",[triggering],"[triggering, triggering中心, triggering中心, trigg...","New Spelling Bee puzzle!\nAllowed letters: A, ...",NaN,NaN,NaN,NaN
4,spelling_bee,4,5.0,0.0,0.0,0.0,i,eghiltw\nc,[],"[lighter, lighter, lighter, lighter, lighter, ...",[lighter],"[lighter, lighter, lighter, lighter, lighter, ...","New Spelling Bee puzzle!\nAllowed letters: E, ...",NaN,NaN,NaN,NaN
